# RAG手法比較ラボ（Google Colab）

このノートブックは `multilingual-e5-small` をColab GPUで実行します。実行前に、Colabの **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選んでください。

In [ ]:
!git clone https://github.com/wataru-i-0823/rag-method-benchmark.git /content/rag-method-benchmark
%cd /content/rag-method-benchmark/work
!pip -q install uv
!uv venv --python 3.11
!uv sync --extra colab

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/rag-method-lab')
for name in ('raw', 'processed', 'mlruns'):
    (DRIVE_ROOT / name).mkdir(parents=True, exist_ok=True)

# 最初の一回だけサンプルをDriveへ配置する。自分のJSONLで置き換えてよい。
for filename in ('example_corpus.jsonl', 'example_qa.jsonl'):
    destination = DRIVE_ROOT / 'processed' / filename
    if not destination.exists():
        shutil.copy2(Path('data') / filename, destination)
print(DRIVE_ROOT)

In [ ]:
# E5 + Chromaと、ローカルベースラインを同じコーパスで比較
!uv run python -m rag_lab evaluate \
  --corpus /content/drive/MyDrive/rag-method-lab/processed/example_corpus.jsonl \
  --qa /content/drive/MyDrive/rag-method-lab/processed/example_qa.jsonl \
  --method bm25,dense,hybrid,chroma_e5 \
  --k 3 --mlflow \
  --tracking-uri file:///content/drive/MyDrive/rag-method-lab/mlruns \
  --experiment e5-colab

In [ ]:
# 結果CSV/JSONをGoogle Driveに保存し、その場で比較表を表示する。GitHubへは追加しない。
!mkdir -p /content/drive/MyDrive/rag-method-lab/results
!cp -f results/* /content/drive/MyDrive/rag-method-lab/results/
import json
import pandas as pd
from IPython.display import display

summary = json.loads(Path('results/summary.json').read_text())
display(pd.DataFrame(summary).T.round(3))